*© Sabrine Bendimerad · [sbendimerad.github.io](https://sbendimerad.github.io) · Licence [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/): you may share and adapt this material with credit, for non-commercial use, under the same licence. Datasets keep their own licences.*

# Ensemble methods on Titanic 🚢🚢

## Introduction

This exercise is an opportunity to practice ensemble methods on a dataset you have worked with before: the Titanic dataset.

Let's start by importing the libraries we will use in the exercise.

In [ ]:
# Load in our libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
# import ensemble methods
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

## Feature Exploration, Engineering and Cleaning 

1. Import the data from the following path: "data/titanic.csv", and display the first lines. Are there any missing values in the dataset?

In [ ]:
# Load in the train and test datasets
data = pd.read_csv("data/titanic.csv")

data.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [ ]:
data.describe(include = 'all')

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,G6,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


In [ ]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


**The following columns have missing values, so we'll have to choose an imputation technique to handle them: Age, Cabin, Embarked**

2. What types of variables are present in this dataset? What kind of preprocessing could you run on these variables?

**The dataset is a mix of numeric and categorical variables. The quantitative variables have to be normalized, and the categorical ones have to be encoded.**

3. Here are some guidelines you can follow to clean the dataset and to create new variables (feature engineering).

a.  Create a Name_length variable that contains the number of characters of the Name variable for each observation.

In [ ]:
data['Name_length'] = data['Name'].apply(len)
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Name_length
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,23
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,51
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,22
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,44
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,24


b. Create a variable Has_Cabin that indicates whether the passenger has a cabin or not.

Hint: [this method](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.notna.html#pandas.DataFrame.notna) might be useful 😉

In [ ]:
# Feature that tells whether a passenger had a cabin on the Titanic
data['Has_Cabin'] = data["Cabin"].notna()
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Name_length,Has_Cabin
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,23,False
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,51,True
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,22,False
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,44,True
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,24,False


c. Create a variable FamilySize that gives the size of each passenger's family.

In [ ]:
# Create new feature FamilySize as a combination of SibSp and Parch
data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Name_length,Has_Cabin,FamilySize
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,23,False,2
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,51,True,2
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,22,False,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,44,True,2
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,24,False,1


d. Create a variable IsAlone that indicates whether the passenger is traveling on their own.

In [ ]:
data['IsAlone'] = False
data.loc[data['FamilySize'] == 1, 'IsAlone'] = True
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Name_length,Has_Cabin,FamilySize,IsAlone
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,23,False,2,False
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,51,True,2,False
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,22,False,1,True
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,44,True,2,False
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,24,False,1,True


e. Extract the title of each passenger to create a variable Title.

Hint: You might consider _applying_ a function that calls the [str.split method](https://docs.python.org/3.3/library/stdtypes.html?highlight=split#str.split) 😉

In [ ]:
# Define function to extract titles from passenger names
def get_title(name):
    return name.split(", ")[1].split(".")[0]

# Create a new feature Title, containing the titles of passenger names
data['Title'] = data['Name'].apply(get_title)
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Name_length,Has_Cabin,FamilySize,IsAlone,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,23,False,2,False,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,51,True,2,False,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,22,False,1,True,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,44,True,2,False,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,24,False,1,True,Mr


f. If some of these titles are equivalent, convert them so that they all fall into the same category.

In [ ]:
px.bar(data, 'Title')

In [ ]:
data['Title'] = data['Title'].replace('Mlle', 'Miss')
data['Title'] = data['Title'].replace('Ms', 'Miss')
data['Title'] = data['Title'].replace('Mme', 'Mrs')
px.bar(data, 'Title')

g. Are any of the remaining titles underrepresented among the observations? If so, group them into a single modality "Rare"

In [ ]:
# Group all non-common titles into one single grouping "Rare"
rare_titles = ['Don', 'Mme', 'Ms', 'Major', 'Lady', 'Sir', 'Mlle', 'Col', 'Capt', 'the Countess', 'Jonkheer']
data['Title'] = data['Title'].replace(rare_titles, 'Rare')
px.bar(data, 'Title')

4. Drop the columns 'PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp' from the dataset. Why don't we need these columns for what comes next?

In [ ]:
# Feature selection
drop_elements = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp']
data = data.drop(drop_elements, axis = 1)

In [ ]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Survived     891 non-null    int64  
 1   Pclass       891 non-null    int64  
 2   Sex          891 non-null    str    
 3   Age          714 non-null    float64
 4   Parch        891 non-null    int64  
 5   Fare         891 non-null    float64
 6   Embarked     889 non-null    str    
 7   Name_length  891 non-null    int64  
 8   Has_Cabin    891 non-null    bool   
 9   FamilySize   891 non-null    int64  
 10  IsAlone      891 non-null    bool   
 11  Title        891 non-null    str    
dtypes: bool(2), float64(2), int64(5), str(3)
memory usage: 71.5 KB


5. Separate the features from the target, and split the data into a train set and a test set (with random_state = 0)

In [ ]:
X = data.drop("Survived", axis=1)
y = data["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify = y, random_state = 0)

6. Using Pipeline and ColumnTransformer, do all the preprocessing at once. Use the [KNN imputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html) to handle the missing values of the numeric variables, and SimpleImputer for the categorical ones.

In [ ]:
# Automatically detect names of numeric/categorical columns
numeric_features = []
categorical_features = []
for i,t in X.dtypes.items():
    if ('float' in str(t)) or ('int' in str(t)) :
        numeric_features.append(i)
    else :
        categorical_features.append(i)

print('Found numeric features ', numeric_features)
print('Found categorical features ', categorical_features)

Found numeric features  ['Pclass', 'Age', 'Parch', 'Fare', 'Name_length', 'FamilySize']
Found categorical features  ['Sex', 'Embarked', 'Has_Cabin', 'IsAlone', 'Title']


In [ ]:
# Create pipeline for numeric features
numeric_transformer = Pipeline(steps=[
    ('imputer', KNNImputer()), 
    ('scaler', StandardScaler())
])

# Create pipeline for categorical features
categorical_transformer = Pipeline(
    steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # missing values will be replaced by most frequent value
    ('encoder', OneHotEncoder(drop='first')) # first column will be dropped to avoid creating correlations between features
    ])

# Use ColumnTransformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = preprocessor.fit_transform(X_train)
print('...Done.')
print(X_train[0:5]) 
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head()) 
X_test = preprocessor.transform(X_test) # Don't fit again !! 
print('...Done.')
print(X_test[0:5,:])

Performing preprocessings on train set...
     Pclass     Sex   Age  Parch    Fare Embarked  Name_length  Has_Cabin  \
671       1    male  31.0      0  52.000        S           22       True   
417       2  female  18.0      2  13.000        S           29      False   
634       3  female   9.0      2  27.900        S           18      False   
323       2  female  22.0      1  29.000        S           51      False   
379       3    male  19.0      0   7.775        S           27      False   

     FamilySize  IsAlone Title  
671           2    False    Mr  
417           3    False  Miss  
634           6    False  Miss  
323           3    False   Mrs  
379           1     True    Mr  
...Done.
[[-1.54562952  0.08147887 -0.46398808  0.35047578 -0.52725961  0.08547193
   1.          0.          1.          1.          0.          0.
   0.          1.          0.          0.          0.        ]
 [-0.35613584 -0.89636308  2.08699167 -0.37942193  0.23157878  0.72699155
   0.      

### Pearson Correlation Heatmap

7. Plot the correlation table of all the explanatory variables of X_train. What do you think?

In [ ]:
# Correlation matrix
corr_matrix = pd.DataFrame(X_train).corr().round(2)

fig = px.imshow(corr_matrix, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Correlation matrix")
fig.show()

**Correlations between the variables are not very high, so we can hope that each of them will bring complementary information to predict the target variable.**

## Ensembling & Stacking models

Now that we have finished the preprocessing and made sure our data is ready for prediction, let's move on to building our ensemble models. We'll train different models with different ensembling strategies and store their train and test scores for comparison.

### Random Forest
8. Train a Random Forest by tuning the hyperparameters with a grid search. Which ensemble method is related to random forests?

Evaluate the best model's accuracy on the train and test sets. Save the scores into a pandas DataFrame.

In [ ]:
scores_df = pd.DataFrame(columns = ['model', 'accuracy', 'set'])

In [ ]:
# Perform grid search
print("Grid search...")
random_forest = RandomForestClassifier()

# Grid of values to be tested
params = {
    'max_depth': [2, 4, 6, 8, 10],
    'min_samples_leaf': [1, 2, 5],
    'min_samples_split': [2, 4, 8],
    'n_estimators': [10, 20, 40, 60, 80, 100]
}
print(params)
gridsearch = GridSearchCV(random_forest, param_grid = params, cv = 3, verbose = 1) # cv : the number of folds to be used for CV
gridsearch.fit(X_train, y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, y_test))
scores_df.loc[len(scores_df)] = {'model': 'random_forest', 'accuracy': gridsearch.score(X_train, y_train), 'set': 'train'}
scores_df.loc[len(scores_df)] = {'model': 'random_forest', 'accuracy': gridsearch.score(X_test, y_test), 'set': 'test'}
scores_df

Grid search...
{'max_depth': [2, 4, 6, 8, 10], 'min_samples_leaf': [1, 2, 5], 'min_samples_split': [2, 4, 8], 'n_estimators': [10, 20, 40, 60, 80, 100]}
Fitting 3 folds for each of 270 candidates, totalling 810 fits


...Done.
Best hyperparameters :  {'max_depth': 8, 'min_samples_leaf': 1, 'min_samples_split': 8, 'n_estimators': 20}
Best validation accuracy :  0.8263308151200528

Accuracy on training set :  0.9026946107784432
Accuracy on test set :  0.8475336322869955


,model,accuracy,set
0,random_forest,0.902695,train
1,random_forest,0.847534,test


9. Create your own Bagging of decision trees (with the same hyperparameters as the optimal ones for the Random Forest) and check that you get similar performance.

In [ ]:
print('Training Bagging of decision tree...')
decision_tree = DecisionTreeClassifier(max_depth = 8, min_samples_leaf = 1, min_samples_split = 4) # Same hyperparameters as best RF model
bagging = BaggingClassifier(estimator=decision_tree, n_estimators = 20)
bagging.fit(X_train, y_train)
print("...Done.")
print("Accuracy on training set : ", bagging.score(X_train, y_train))
print("Accuracy on test set : ", bagging.score(X_test, y_test))
scores_df.loc[len(scores_df)] = {'model': 'bagging_dt', 'accuracy': bagging.score(X_train, y_train), 'set': 'train'}
scores_df.loc[len(scores_df)] = {'model': 'bagging_dt', 'accuracy': bagging.score(X_test, y_test), 'set': 'test'}
scores_df

Training Bagging of decision tree...
...Done.
Accuracy on training set :  0.9431137724550899
Accuracy on test set :  0.8161434977578476


,model,accuracy,set
0,random_forest,0.902695,train
1,random_forest,0.847534,test
2,bagging_dt,0.943114,train
3,bagging_dt,0.816143,test


**The performances are not exactly the same! However, remember that this dataset is quite small, so the difference is actually not significant 😌 (if you're curious: use a cross-validation to estimate the uncertainty on the test scores)**

10. Train an AdaBoost model by tuning the hyperparameters:
* With a logistic regression as base estimator
* With a decision tree as base estimator

For each model, evaluate the performance on the test set.

In [ ]:
# Perform grid search
print("Grid search...")
logistic_regression = LogisticRegression(max_iter = 1000)
adaboost_logreg = AdaBoostClassifier(logistic_regression)

# Grid of values to be tested
params = {
    'estimator__C': [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0],
    'n_estimators': [5, 10, 20, 40, 60, 80, 100]
}
print(params)
gridsearch = GridSearchCV(adaboost_logreg, param_grid = params, cv = 3, verbose = 1) # cv : the number of folds to be used for CV
gridsearch.fit(X_train, y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, y_test))
scores_df.loc[len(scores_df)] = {'model': 'adaboost_logreg', 'accuracy': gridsearch.score(X_train, y_train), 'set': 'train'}
scores_df.loc[len(scores_df)] = {'model': 'adaboost_logreg', 'accuracy': gridsearch.score(X_test, y_test), 'set': 'test'}
scores_df

Grid search...
{'estimator__C': [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0], 'n_estimators': [5, 10, 20, 40, 60, 80, 100]}
Fitting 3 folds for each of 56 candidates, totalling 168 fits


...Done.
Best hyperparameters :  {'estimator__C': 50.0, 'n_estimators': 40}
Best validation accuracy :  0.8323031551731104

Accuracy on training set :  0.8308383233532934
Accuracy on test set :  0.8116591928251121


,model,accuracy,set
0,random_forest,0.902695,train
1,random_forest,0.847534,test
2,bagging_dt,0.943114,train
3,bagging_dt,0.816143,test
4,adaboost_logreg,0.830838,train
5,adaboost_logreg,0.811659,test


In [ ]:
# Perform grid search
print("Grid search...")
decision_tree = DecisionTreeClassifier()
adaboost_dt = AdaBoostClassifier(decision_tree)

# Grid of values to be tested
params = {
    'estimator__max_depth': [8, 10, 12],
    'estimator__min_samples_leaf': [1, 2, 3],
    'estimator__min_samples_split': [6, 8, 10],
    'n_estimators': [2, 4, 6, 8, 10, 12]
}
print(params)
gridsearch = GridSearchCV(adaboost_dt, param_grid = params, cv = 3, verbose = 1) # cv : the number of folds to be used for CV
gridsearch.fit(X_train, y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, y_test))
scores_df.loc[len(scores_df)] = {'model': 'adaboost_dt', 'accuracy': gridsearch.score(X_train, y_train), 'set': 'train'}
scores_df.loc[len(scores_df)] = {'model': 'adaboost_dt', 'accuracy': gridsearch.score(X_test, y_test), 'set': 'test'}
scores_df

Grid search...
{'estimator__max_depth': [8, 10, 12], 'estimator__min_samples_leaf': [1, 2, 3], 'estimator__min_samples_split': [6, 8, 10], 'n_estimators': [2, 4, 6, 8, 10, 12]}
Fitting 3 folds for each of 162 candidates, totalling 486 fits


...Done.
Best hyperparameters :  {'estimator__max_depth': 8, 'estimator__min_samples_leaf': 1, 'estimator__min_samples_split': 10, 'n_estimators': 4}
Best validation accuracy :  0.7948868150661873

Accuracy on training set :  0.9505988023952096
Accuracy on test set :  0.820627802690583


,model,accuracy,set
0,random_forest,0.902695,train
1,random_forest,0.847534,test
2,bagging_dt,0.943114,train
3,bagging_dt,0.816143,test
4,adaboost_logreg,0.830838,train
5,adaboost_logreg,0.811659,test
6,adaboost_dt,0.950599,train
7,adaboost_dt,0.820628,test


11. Train scikit-learn's GradientBoosting model (with hyperparameter tuning) and evaluate its performance.

In [ ]:
# Perform grid search
print("Grid search...")
gradientboost = GradientBoostingClassifier()

# Grid of values to be tested
params = {
    'max_depth': [8, 10, 12],
    'min_samples_leaf': [1, 2, 3],
    'min_samples_split': [6, 8, 10],
    'n_estimators': [2, 4, 6, 8, 10, 12]
}
print(params)
gridsearch = GridSearchCV(gradientboost, param_grid = params, cv = 3, verbose = 1) # cv : the number of folds to be used for CV
gridsearch.fit(X_train, y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, y_test))
scores_df.loc[len(scores_df)] = {'model': 'gradient_boost', 'accuracy': gridsearch.score(X_train, y_train), 'set': 'train'}
scores_df.loc[len(scores_df)] = {'model': 'gradient_boost', 'accuracy': gridsearch.score(X_test, y_test), 'set': 'test'}
scores_df

Grid search...
{'max_depth': [8, 10, 12], 'min_samples_leaf': [1, 2, 3], 'min_samples_split': [6, 8, 10], 'n_estimators': [2, 4, 6, 8, 10, 12]}
Fitting 3 folds for each of 162 candidates, totalling 486 fits


...Done.
Best hyperparameters :  {'max_depth': 8, 'min_samples_leaf': 3, 'min_samples_split': 10, 'n_estimators': 10}
Best validation accuracy :  0.8248091140467823

Accuracy on training set :  0.9251497005988024
Accuracy on test set :  0.8295964125560538


,model,accuracy,set
0,random_forest,0.902695,train
1,random_forest,0.847534,test
2,bagging_dt,0.943114,train
3,bagging_dt,0.816143,test
4,adaboost_logreg,0.830838,train
5,adaboost_logreg,0.811659,test
6,adaboost_dt,0.950599,train
7,adaboost_dt,0.820628,test
8,gradient_boost,0.925150,train
9,gradient_boost,0.829596,test


12. Train an XGBoost model (with hyperparameter tuning). Do you get better or similar results compared to scikit-learn's GradientBoosting?

In [ ]:
# Perform grid search
print("Grid search...")
xgboost = XGBClassifier()

# Grid of values to be tested
params = {
    'max_depth': [4, 6, 8, 10],
    'min_child_weight': [1, 2, 4, 6, 8],
    'n_estimators': [2, 4, 6, 8, 10, 12]
}
print(params)
gridsearch = GridSearchCV(xgboost, param_grid = params, cv = 3, verbose = 1) # cv : the number of folds to be used for CV
gridsearch.fit(X_train, y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, y_test))
scores_df.loc[len(scores_df)] = {'model': 'xgboost', 'accuracy': gridsearch.score(X_train, y_train), 'set': 'train'}
scores_df.loc[len(scores_df)] = {'model': 'xgboost', 'accuracy': gridsearch.score(X_test, y_test), 'set': 'test'}
scores_df

Grid search...
{'max_depth': [4, 6, 8, 10], 'min_child_weight': [1, 2, 4, 6, 8], 'n_estimators': [2, 4, 6, 8, 10, 12]}
Fitting 3 folds for each of 120 candidates, totalling 360 fits


...Done.
Best hyperparameters :  {'max_depth': 8, 'min_child_weight': 4, 'n_estimators': 10}
Best validation accuracy :  0.830821853243378

Accuracy on training set :  0.9026946107784432
Accuracy on test set :  0.8116591928251121


,model,accuracy,set
0,random_forest,0.902695,train
1,random_forest,0.847534,test
2,bagging_dt,0.943114,train
3,bagging_dt,0.816143,test
4,adaboost_logreg,0.830838,train
5,adaboost_logreg,0.811659,test
6,adaboost_dt,0.950599,train
7,adaboost_dt,0.820628,test
8,gradient_boost,0.925150,train
9,gradient_boost,0.829596,test


13. Compare the performance of all the models in a bar chart and conclude. Which model is the best?

Hint: the option `barmode` in plotly's `px.bar()` might be useful 😇

In [ ]:
scores_df = scores_df.sort_values(by = ['set', 'accuracy'], ascending = False)
scores_df

,model,accuracy,set
6,adaboost_dt,0.950599,train
2,bagging_dt,0.943114,train
8,gradient_boost,0.925150,train
0,random_forest,0.902695,train
10,xgboost,0.902695,train
4,adaboost_logreg,0.830838,train
1,random_forest,0.847534,test
9,gradient_boost,0.829596,test
7,adaboost_dt,0.820628,test
3,bagging_dt,0.816143,test


In [ ]:
px.bar(scores_df, x = "model", y = 'accuracy', color = 'set', barmode = 'group', width = 900)

**The gradient boosting model seems to be a bit better than the others: it has the best test accuracy and still doesn't overfit much.**